### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="thyroid_discordant",
    dataset_year="1986",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5D010",
    download_description="""
We download the dis.data from the UCI files. All other files are corrupted or versions of the same data as far as we know (see the dataset curation sheet for more information).

wget https://archive.ics.uci.edu/static/public/102/thyroid+disease.zip && unzip thyroid+disease.zip  dis.data dis.test && rm thyroid+disease.zip && mkdir -p local-data-warehouse/thyroid_discordant && mv dis.data dis.test local-data-warehouse/thyroid_discordant/
""",
    # References, we cite this version as we found a reference to this dataset inside of it.
    # We did not find a PDF for the other reference.
    academic_reference_bibtex="""@article{quinlan1987simplifying,
  title={Simplifying decision trees},
  author={Quinlan, J. Ross},
  journal={International journal of man-machine studies},
  volume={27},
  number={3},
  pages={221--234},
  year={1987},
  publisher={Elsevier}
}
""",
    academic_reference_bibtex_key="quinlan1987simplifying",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
We start with the .data file from UCI and add columns and encode nan values.

- Note, the data contains NaN indicators by default (e.g. TSH_measured). We keep them.
- We drop/ignore the referral_source columns as it indicates the original of the data and thus might leak information or makes the model learn sub-group related behavior that is not the target/task of interest.
- We correct the label column by removing the patient IDs.
- We drop 60 duplicated rows and remove constant columns.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="discordant",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="discordant",
)

## Preprocessing

In [ ]:
import pandas as pd

columns = [
    "age","sex","on_thyroxine","query_on_thyroxine","on_antithyroid_medication","sick","pregnant","thyroid_surgery","I131_treatment","query_hypothyroid","query_hyperthyroid","lithium","goitre","tumor","hypopituitary","psych","TSH_measured","TSH","T3_measured","T3","TT4_measured","TT4","T4U_measured","T4U","FTI_measured","FTI","TBG_measured","TBG","referral_source","discordant",
]
df = pd.read_csv(dataset_mold.path / "dis.data", header=None, names=columns, na_values=["?"])
df = pd.concat([df, pd.read_csv(dataset_mold.path / "dis.test", header=None, names=columns, na_values=["?"])], ignore_index=True)
print("Loaded data shape:", df.shape)

df["discordant"] = df["discordant"].str.split(".").str[0] # remove patient ID from target column

as_cat_type = [
    "discordant","sex","on_thyroxine","query_on_thyroxine","on_antithyroid_medication","thyroid_surgery","query_hypothyroid","query_hyperthyroid","pregnant","sick","tumor","lithium","goitre","TSH_measured","T3_measured","TT4_measured","T4U_measured","FTI_measured","TBG_measured", "psych", "I131_treatment", "hypopituitary",
]
df[as_cat_type] = df[as_cat_type].astype("category")

df = df.drop(columns=[
    "referral_source", # irrelevant here
    "TBG_measured", "TBG", # constant
])

# drop duplicated rows
df = df.drop_duplicates().reset_index(drop=True)

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
        random_state=split_random_state,
    ),
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()